In [1]:
import pandas as pd
import torchaudio
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModelForAudioClassification
import torch
import numpy as np
from sklearn.preprocessing import StandardScaler

C:\Users\castl\anaconda3\envs\pytorch_p311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
model_name = "Heem2/Deepfake-audio-detection"
# processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForAudioClassification.from_pretrained(model_name).to(device)

In [26]:
class AudioDatasets(Dataset):
    def __init__(self, dataframe, is_train = True):
        self.dataframe = dataframe
        self.is_train = is_train
    
    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        audio_path = self.dataframe.iloc[idx]['path']
        waveform, sample_rate = torchaudio.load(audio_path)

        if sample_rate != 16000:
            transform = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
            waveform = transform(waveform)

        waveform = waveform.mean(dim=0)

        if self.is_train:
            label = 1 if self.dataframe.iloc[idx]['label'] == 'fake' else 0
            return waveform, label
        else:
            return waveform

In [27]:
def collate_fn(batch):
    waveforms = [item[0] for item in batch]
    if len(batch[0]) == 2:
        labels = torch.tensor([item[1] for item in batch])
    else:
        labels = None

    max_length = max([waveform.size(0) for waveform in waveforms])
    padded_waveforms = [torch.nn.functional.pad(waveform, (0, max_length - waveform.size(0))) for waveform in waveforms]
    padded_waveforms = torch.stack(padded_waveforms)

    if labels is not None:
        return padded_waveforms, labels
    else:
        return padded_waveforms

In [28]:
train_dataset = AudioDatasets(train_df)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn = collate_fn)

In [29]:
test_dataset = AudioDatasets(test_df)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=True, collate_fn = collate_fn)

In [30]:
import torch.optim as optim
import torch.nn.functional as F

optimizer = optim.Adam(model.parameters(), lr = 2e-5)

In [39]:
from tqdm import tqdm

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix

In [40]:
def train(model, train_lodaer, optimizer, device):
    model.train()
    total_loss = 0.0
    all_labels = []
    all_preds = []
    for batch in tqdm(train_loader):
        inputs, labels = batch
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = F.cross_entropy(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        all_lables.extend(labels.cpu().numpy())
        all_preds.extend(outputs.logits.argmax(dim=1).cpu().numpy())

    avg_loss = total_loss / len(train_loader)
    auc = roc_auc_score(all_labels, all_preds)
    f1 = f1_score(all_lables, all_preds)
    cm = confusion_matrix(all_labels, all_preds)
    return avg_loss, auc, f1, cm

In [41]:
for epoch in range(10):
    print(f"Epoch {epoch+1}")
    avg_loss, auc, f1, cm = train(model, train_loader, optimizer, device)
    print(f"loss: {avg_loss:.4f}, AUC: {auc:.4f}, F1 Score: {f1:.4f}")

epoch:0


  1%|▌                                                                            | 52/6930 [12:54<28:27:40, 14.90s/it]


KeyboardInterrupt: 

In [ ]:
model.eval()

results = []

for batch in test_loader:
    inputs = batch.to(device)
    with torch.no_grad():
        outputs = model(inputs)
    logits = outputs.logits
    probs = torch.nn.functional.softmax(logits, dim = -1)
    fakes = probs[:,1].cpu().numpy()
    reals = probs[:,0].cpu().numpy()

    for i in range(len(inputs)):
        results.append([test_df.iloc[i]['id'], fakes[i], reals[i]])

submission_df = pd.DataFrame(results, columns = ['id', 'fake', 'real'])
submission_df.to_csv('/results/submission.csv', index = False)